# Training the PlantVillage disease classifier

Drives `ml/train.py` — the logic lives there so training is reproducible from the command
line and from CI, not only from a notebook. This page is the control panel and the report.

**Two-stage transfer learning.** Stage 1 freezes the ResNet-50 backbone and trains only the
new 38-class head; a randomly-initialised head produces large early gradients that would
otherwise wreck the pretrained features. Stage 2 unfreezes `layer4` and fine-tunes at a
tenth of the learning rate.

**Why the augmentation is aggressive on colour.** PlantVillage is shot in a lab under
uniform lighting on a uniform background. Without heavy photometric augmentation the model
learns the *lighting* and collapses on a phone photo taken in a field. Colour jitter matters
more here than geometry.

In [ ]:
from _bootstrap import setup

ROOT = setup()

import json
import subprocess
import sys as _sys

import matplotlib.pyplot as plt
import torch
import yaml

CONFIG_PATH = ROOT / "ml" / "configs" / "resnet50_plantvillage.yaml"
config = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))

print(f"torch  : {torch.__version__}")
print(f"cuda   : {torch.cuda.is_available()}"
      + (f" ({torch.cuda.get_device_name(0)})" if torch.cuda.is_available() else " — CPU training will be slow"))
print(f"config : {CONFIG_PATH.relative_to(ROOT)}\n")
print(yaml.dump(config, sort_keys=False))

## 1. Is the dataset there?

Expected as an `ImageFolder` tree at `data/raw/plantvillage/<Crop>___<Disease>/*.jpg`.
Get it with `scripts/download_plantvillage.py --source kaggle` (free account + API token),
or follow `--source manual` for the browser route.

In [ ]:
data_dir = ROOT / config["data_dir"]

if not data_dir.exists():
    print(f"NOT FOUND: {data_dir}")
    print("\nRun:  python scripts/download_plantvillage.py --source kaggle")
    print("The rest of this notebook needs the dataset; the pipeline notebook does not.")
    DATASET_READY = False
else:
    class_dirs = sorted(p for p in data_dir.iterdir() if p.is_dir())
    counts = {p.name: sum(1 for _ in p.glob("*.[jJpP]*[gG]")) for p in class_dirs}
    total = sum(counts.values())
    DATASET_READY = total > 0

    print(f"{len(class_dirs)} classes, {total:,} images\n")
    ordered = sorted(counts.items(), key=lambda kv: kv[1])
    print("smallest classes:")
    for name, count in ordered[:5]:
        print(f"  {count:6,}  {name}")
    print("largest classes:")
    for name, count in ordered[-5:]:
        print(f"  {count:6,}  {name}")
    print(f"\nimbalance ratio (max/min): {ordered[-1][1] / max(ordered[0][1], 1):.1f}x")
    print("=> report macro F1, not just accuracy — see ml/evaluate.py")

In [ ]:
if DATASET_READY:
    figure, ax = plt.subplots(figsize=(11, 7))
    names = [name for name, _ in ordered]
    values = [count for _, count in ordered]
    ax.barh(range(len(names)), values, color="#2e7d32")
    ax.set_yticks(range(len(names)), names, fontsize=6)
    ax.set_xlabel("images")
    ax.set_title("PlantVillage class distribution")
    figure.tight_layout()

## 2. Inspect the augmentation policy

Look at this before spending GPU hours. If the augmented crops no longer show the lesion,
the policy is too aggressive and the model cannot learn what you want it to learn.

In [ ]:
if DATASET_READY:
    from datasets import build_transforms
    from torchvision.datasets import ImageFolder

    train_transform = build_transforms(config, train=True)
    raw = ImageFolder(str(data_dir))
    index = next(i for i, (_, label) in enumerate(raw.samples) if label == 0)
    original, _ = raw[index]

    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

    figure, axes = plt.subplots(2, 5, figsize=(15, 6.5))
    axes.ravel()[0].imshow(original)
    axes.ravel()[0].set_title(f"original\n{raw.classes[0]}", fontsize=8)
    axes.ravel()[0].axis("off")
    for ax in axes.ravel()[1:]:
        tensor = train_transform(original) * std + mean
        ax.imshow(tensor.permute(1, 2, 0).clamp(0, 1).numpy())
        ax.set_title("augmented", fontsize=8)
        ax.axis("off")
    figure.tight_layout()

## 3. Train

This shells out to `ml/train.py` so the notebook and the CLI cannot diverge. On CPU, drop
`epochs_head` to 1 and `epochs_finetune` to 1 first to confirm the loop runs end to end
before committing to a real run.

In [ ]:
SMOKE_TEST = True    # flip to False for the real run

command = [
    _sys.executable, str(ROOT / "ml" / "train.py"),
    "--config", str(CONFIG_PATH),
]
if SMOKE_TEST:
    command += ["--epochs-head", "1", "--epochs-finetune", "0", "--batch-size", "16"]

print(" ".join(command), "\n")

if DATASET_READY:
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, cwd=str(ROOT), bufsize=1)
    for line in process.stdout:
        print(line, end="")
    print(f"\nexit code: {process.wait()}")
else:
    print("Dataset missing — skipping.")

## 4. Learning curves

In [ ]:
history_path = ROOT / "models" / "checkpoints" / "resnet50_plantvillage_history.json"

if history_path.exists():
    history = json.loads(history_path.read_text(encoding="utf-8"))["history"]
    epochs = range(1, len(history) + 1)

    figure, (left, right) = plt.subplots(1, 2, figsize=(13, 4.5))
    left.plot(epochs, [entry["train_loss"] for entry in history], marker="o", label="train")
    left.plot(epochs, [entry["val_loss"] for entry in history], marker="o", label="val")
    left.set_title("loss")
    left.set_xlabel("epoch")
    left.legend()
    left.grid(alpha=0.2)

    right.plot(epochs, [entry["train_acc"] for entry in history], marker="o", label="train")
    right.plot(epochs, [entry["val_acc"] for entry in history], marker="o", label="val")
    right.set_title("accuracy")
    right.set_xlabel("epoch")
    right.legend()
    right.grid(alpha=0.2)

    for ax in (left, right):
        boundary = sum(1 for entry in history if entry["stage"] == "head")
        if 0 < boundary < len(history):
            ax.axvline(boundary + 0.5, color="grey", linestyle="--", linewidth=1)
    figure.tight_layout()

    best = max(entry["val_acc"] for entry in history)
    print(f"best val accuracy: {best:.4f}")
else:
    print(f"No history yet at {history_path.name} — train first.")

## 5. Evaluate

`ml/evaluate.py` writes a per-class report and a row-normalised confusion matrix. Per-class
F1 is the number that matters: with a 20x class imbalance, a model that nails the big tomato
classes and fails a small one still posts a flattering headline accuracy.

In [ ]:
checkpoint = ROOT / config["output_path"]

if checkpoint.exists():
    process = subprocess.run(
        [_sys.executable, str(ROOT / "ml" / "evaluate.py"), "--checkpoint", str(checkpoint)],
        capture_output=True, text=True, cwd=str(ROOT),
    )
    print(process.stdout[-4000:])
    if process.returncode:
        print("STDERR:", process.stderr[-1500:])
else:
    print(f"No checkpoint at {checkpoint} — train first.")

## 6. Wire it into the pipeline

Once the checkpoint exists, nothing needs changing: `app.vision.classifier` picks it up from
`CHECKPOINT_PATH` and the readiness matrix flips `checkpoint_loaded` to `true`. The
`confidence_reliable` flag and the PROVISIONAL prompt warning disappear on their own.

The class list is saved *inside* the checkpoint and read back at inference. That is the only
thing preventing a silent label-order mismatch between training and serving — the failure
mode where every prediction is confidently wrong.

In [ ]:
from app.config import get_settings
from app.vision.classifier import DiseaseClassifier
from app.vision.preprocess import read_image

settings = get_settings()
classifier = DiseaseClassifier(settings)
print(f"checkpoint loaded : {classifier.status.checkpoint_loaded}")
print(f"classes           : {len(classifier.classes)}")
if classifier.status.note:
    print(f"note              : {classifier.status.note}")

sample = ROOT / "data" / "samples" / "leaf_moderate.png"
if sample.exists():
    diagnosis = classifier.predict(read_image(str(sample)), top_k=3)
    print(f"\n{diagnosis.top.crop} / {diagnosis.top.disease}  {diagnosis.top.confidence:.1%}"
          f"   reliable={diagnosis.confidence_reliable}")